**Лабораторная работа № 1 «Предварительный анализ данных»**

Дисциплина: «Анализ данных». Выполнил: Жойд Станислав Вадимович, группа 4417, вариант 8.
Преподаватель: Боженко Виктория Вячеславовна, старший преподаватель кафедры 41.

**Цель работы:** осуществить предварительную обработку данных CSV-файла средствами pandas, выявить и устранить обнаруженные проблемы.

# Загрузка набора данных

### Описание предметной области

Вариант № 8. Набор данных: **credit_risk.csv**, предоставленный вместе с заданием.
Предметная область — анализ кредитных заявок: сведения о заёмщиках, целях и параметрах кредита и кредитной истории.

| Столбец исходного файла | Содержание |
|---|---|
| `Id` | Идентификатор записи; не количественная характеристика заёмщика. |
| `Age` | Возраст заёмщика, лет. |
| `Income` | Доход заёмщика; денежная единица в предоставленном задании не уточнена. |
| `Home` | Статус домовладения: RENT — аренда, OWN — собственное жильё, MORTGAGE — ипотека, OTHER — другое. |
| `Emp_length` | Продолжительность трудового стажа, лет. |
| `Intent` | Цель кредита: EDUCATION — образование, MEDICAL — медицинские расходы, VENTURE — предпринимательство, PERSONAL — личные нужды, DEBTCONSOLIDATION — объединение долгов, HOMEIMPROVEMENT — улучшение жилья. |
| `Amount` | Сумма кредита; денежная единица не уточнена. |
| `Rate` | Процентная ставка по кредиту. |
| `Status` | Статус кредита, закодированный значениями 0 и 1; соответствие кодов конкретным состояниям в задании не указано. |
| `Percent_income` | Доля суммы кредита в доходе: например, 0,59 соответствует 59 %. Обнаруженные расхождения с отношением Amount/Income рассмотрены далее. |
| `Default` | Признак дефолта в кредитной истории: Y — да, N — нет. |
| `Cred_length` | Продолжительность кредитной истории, лет. |

Признаки `Status` и `Default` описывают разные характеристики; совпадение их кодов не требуется.

### 1.Чтение файла (набора данных)

Для локального выполнения следует открыть этот ноутбук в папке лабораторной работы и сохранить исходный файл в `utility/credit_risk.csv` либо рядом с ноутбуком.
В Google Colab при отсутствии файла появится окно загрузки: необходимо выбрать предоставленный **credit_risk.csv** на компьютере. Автоматическая загрузка данных из сети не выполняется.
Объект `raw_df` сохраняет исходный набор без изменений; предобработка выполняется в отдельной копии `df`.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.2f}'.format)

dataset_path = next((path for path in [Path('credit_risk.csv'), Path('utility/credit_risk.csv')]
                     if path.is_file()), None)
if dataset_path is None:
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Поместите credit_risk.csv рядом с ноутбуком или в папку utility.')
    uploaded = files.upload()
    dataset_path = Path('credit_risk.csv')
    if not dataset_path.is_file():
        raise FileNotFoundError('Необходимо загрузить файл с именем credit_risk.csv.')

raw_df = pd.read_csv(dataset_path)
df = raw_df.copy(deep=True)
print(f'Размер исходного набора: {df.shape[0]} строк, {df.shape[1]} столбцов.')

Размер исходного набора: 652 строк, 12 столбцов.


### 2. Обзор данных

2.1 Вывод первых 20 строк с помощью метода `head`. Первые наблюдения позволяют проверить чтение файла и увидеть типичные значения признаков.

In [2]:
display(df.head(20))

,Id,Age,Income,Home,Emp_length,Intent,Amount,Rate,Status,Percent_income,Default,Cred_length
0,0,22.00,59000,RENT,123.00,PERSONAL,35000,16.02,1,0.59,Y,3
1,1,21.00,9600,OWN,5.00,EDUCATION,1000,11.14,0,0.10,N,2
2,2,25.00,9600,MORTGAGE,1.00,MEDICAL,5500,12.87,1,0.57,N,3
3,3,23.00,65500,RENT,4.00,MEDICAL,35000,15.23,1,0.53,N,2
4,4,24.00,54400,RENT,8.00,MEDICAL,35000,14.27,1,0.55,Y,4
5,5,21.00,9900,OWN,2.00,VENTURE,2500,7.14,1,0.25,N,2
6,6,26.00,77100,RENET,8.00,EDUCATION,35000,12.42,1,0.45,N,3
7,7,24.00,78956,RENT,5.00,MEDICAL,35000,11.11,1,0.44,N,4
8,8,24.00,83000,RENT,8.00,PERSONAL,35000,8.90,1,0.42,N,2
9,9,21.00,10000,OWN,6.00,VENTURE,1600,14.74,1,0.16,N,3


2.2 Оценка данных с помощью метода `info`: число наблюдений, типы столбцов и количество непустых значений.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 652 entries, 0 to 651
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Id              652 non-null    int64  
 1   Age             652 non-null    float64
 2   Income          652 non-null    int64  
 3   Home            652 non-null    object 
 4   Emp_length      643 non-null    float64
 5   Intent          652 non-null    object 
 6   Amount          652 non-null    int64  
 7   Rate            586 non-null    float64
 8   Status          652 non-null    int64  
 9   Percent_income  652 non-null    float64
 10  Default         652 non-null    object 
 11  Cred_length     652 non-null    int64  
dtypes: float64(4), int64(5), object(3)
memory usage: 61.3+ KB


Загружено 652 строки и 12 столбцов. `Emp_length` содержит 643 непустых значения, `Rate` — 586; в остальных столбцах по 652 значения.
Числовые поля представлены типами `int64` и `float64`, текстовые категории — `object`.
Возраст записан как число с плавающей точкой, хотя дробных возрастов в наборе нет.

2.3 Оценка числовых столбцов с помощью метода `describe`. Среднее идентификатора выводится автоматически, однако содержательно не интерпретируется.

In [4]:
display(df.describe())

,Id,Age,Income,Emp_length,Amount,Rate,Status,Percent_income,Cred_length
count,652.00,652.00,652.00,643.00,652.00,586.00,652.00,652.00,652.00
mean,325.50,24.29,90008.86,4.60,18801.00,12.29,0.60,0.28,3.01
std,188.35,7.84,69530.31,7.28,9152.90,3.27,0.49,0.15,0.81
min,0.00,21.00,9600.00,0.00,1000.00,5.42,0.00,0.01,2.00
25%,162.75,23.00,44000.00,2.00,10000.00,10.25,0.00,0.16,2.00
50%,325.50,24.00,69998.00,4.00,21850.00,12.18,1.00,0.28,3.00
75%,488.25,25.00,128499.00,7.00,25000.00,14.72,1.00,0.38,4.00
max,649.00,144.00,500000.00,123.00,35000.00,21.21,1.00,0.83,4.00


Доход изменяется от 9 600 до 500 000, сумма кредита — от 1 000 до 35 000, ставка — от 5,42 до 21,21.
Максимальный возраст 144 года и максимальный стаж 123 года требуют отдельной проверки. Средние исходного набора пока не используются как окончательный результат: на них влияют пропуски, дубликаты и аномалии.

В качестве дополнительной проверки сопоставлена доля `Percent_income` с `Amount / Income`. Чтобы не считать обычное округление ошибкой, выделены расхождения более 0,011. Найдено 17 строк, все с целью MEDICAL. Это не основание автоматически пересчитывать предоставленный показатель: причина расхождений по файлу не устанавливается. В заданиях варианта данный столбец не агрегируется.

In [5]:
income_share_difference = (raw_df['Percent_income'] - raw_df['Amount'] / raw_df['Income']).abs()
share_discrepancies = raw_df.loc[income_share_difference > 0.011,
                               ['Id', 'Income', 'Intent', 'Amount', 'Percent_income']]
print(f'Число расхождений более 0,011: {len(share_discrepancies)}')
display(share_discrepancies)

Число расхождений более 0,011: 17


,Id,Income,Intent,Amount,Percent_income
4,4,54400,MEDICAL,35000,0.55
17,17,92111,MEDICAL,35000,0.32
40,40,62050,MEDICAL,30000,0.41
48,48,66300,MEDICAL,30000,0.38
63,63,221850,MEDICAL,25000,0.10
202,202,56100,MEDICAL,25000,0.38
224,224,62050,MEDICAL,25000,0.34
260,260,66215,MEDICAL,25000,0.32
270,270,66300,MEDICAL,25000,0.32
278,278,70550,MEDICAL,25000,0.30


2.4 Оценка названий столбцов. Названия непустые, уникальные и не содержат пробелов по краям. Для единообразия далее используется нижний регистр, соответствующий формулировкам заданий.

In [6]:
display(df.columns)

Index(['Id', 'Age', 'Income', 'Home', 'Emp_length', 'Intent', 'Amount', 'Rate',
       'Status', 'Percent_income', 'Default', 'Cred_length'],
      dtype='object')

In [7]:
df.columns = df.columns.str.lower()
display(df.columns)

Index(['id', 'age', 'income', 'home', 'emp_length', 'intent', 'amount', 'rate',
       'status', 'percent_income', 'default', 'cred_length'],
      dtype='object')

### 3. Проверка пропусков

In [8]:
missing_initial = df.isna().sum()
display(missing_initial.rename('Количество пропусков').to_frame())
rows_with_missing = int(df.isna().any(axis=1).sum())
print(f'Строк с хотя бы одним пропуском: {rows_with_missing}')

,Количество пропусков
id,0
age,0
income,0
home,0
emp_length,9
intent,0
amount,0
rate,66
status,0
percent_income,0


Строк с хотя бы одним пропуском: 74


Обнаружено 9 пропусков стажа и 66 пропусков ставки: всего 75 ячеек в 74 строках, поскольку одна строка содержит оба пропуска.
Выбран метод удаления неполных наблюдений (`dropna`), разрешённый заданием. Он не создаёт искусственных значений и обеспечивает прозрачную воспроизводимость результата.
Возможная альтернатива — заполнение медианой, однако медианное значение являлось бы оценкой, а не фактическим наблюдением.

Стаж и ставка непосредственно не входят в четыре требуемые группировки, поэтому удаление строк приводит и к потере пригодных сведений по другим признакам. Случайность пропусков не доказана; полученные оценки характеризуют очищенную подвыборку и могут быть смещены относительно полного набора.

In [9]:
df = df.dropna().copy()
rows_after_dropna = len(df)
print(f'После удаления неполных наблюдений: {rows_after_dropna} строк.')
display(df.isna().sum().rename('Осталось пропусков').to_frame())

После удаления неполных наблюдений: 578 строк.


,Осталось пропусков
id,0
age,0
income,0
home,0
emp_length,0
intent,0
amount,0
rate,0
status,0
percent_income,0


### 4. Проверка дубликатов

#### Проверка явных дубликатов

In [10]:
duplicate_count = int(df.duplicated().sum())
print(f'Лишних полных дубликатов: {duplicate_count}')
display(df.loc[df.duplicated(keep=False)])

Лишних полных дубликатов: 2


,id,age,income,home,emp_length,intent,amount,rate,status,percent_income,default,cred_length
649,649,23.00,58800,RENT,7.00,DEBTCONSOLIDATION,20000,7.66,1,0.34,N,3
650,649,23.00,58800,RENT,7.00,DEBTCONSOLIDATION,20000,7.66,1,0.34,N,3
651,649,23.00,58800,RENT,7.00,DEBTCONSOLIDATION,20000,7.66,1,0.34,N,3


Запись с идентификатором 649 встречается трижды и полностью совпадает по всем признакам. Сохраняется первое вхождение, удаляются две лишние копии. Равенство отдельных признаков разных заёмщиков само по себе не является основанием для удаления.

In [11]:
df = df.drop_duplicates().copy()
rows_after_duplicates = len(df)
print(f'После удаления полных дубликатов: {rows_after_duplicates} строк.')

После удаления полных дубликатов: 576 строк.


#### Проверка неявных дубликатов

In [12]:
for column in ['home', 'intent', 'default']:
    print(f'Значения {column}:')
    display(df[column].value_counts(dropna=False).rename('count').to_frame())

Значения home:


,count
home,
RENT,376
MORTGAGE,114
OWN,83
OTHER,2
RENET,1


Значения intent:


,count
intent,
EDUCATION,128
VENTURE,108
PERSONAL,102
MEDICAL,92
DEBTCONSOLIDATION,87
HOMEIMPROVEMENT,59


Значения default:


,count
default,
N,438
Y,135
No,3


В `home` обнаружена опечатка `RENET`, соответствующая категории `RENT`; в `default` значение `No` совпадает по смыслу с `N`.
В исходном файле это соответственно одна и три записи. Исправление объединяет разные написания одной категории, но не удаляет наблюдения.
В `intent` альтернативных написаний одинаковой цели не обнаружено. Категория `OTHER` допустима и сохраняется.

In [13]:
df['home'] = df['home'].replace({'RENET': 'RENT'})
df['default'] = df['default'].replace({'No': 'N'})
for column in ['home', 'default']:
    display(df[column].value_counts().rename('count').to_frame())
print('Полных дубликатов после нормализации:', df.duplicated().sum())
print('Дубликатов без учёта идентификатора:', df.drop(columns='id').duplicated().sum())

,count
home,
RENT,377
MORTGAGE,114
OWN,83
OTHER,2


,count
default,
N,441
Y,135


Полных дубликатов после нормализации: 0
Дубликатов без учёта идентификатора: 0


### 5. Провека типов данных

Перед приведением типов проверяются аномальные числовые значения, обнаруженные при обзоре. Для этого исследования выбран критерий исключения `age > 100` либо `emp_length > age`. Порог 100 лет не является универсальной границей возраста человека: в данном файле он отделяет три значения 123 и 144 от остальных возрастов 21–26. Стаж 123 года у заёмщиков 21 и 22 лет невозможен. Достоверные исправления неизвестны, поэтому исключаются соответствующие строки.

In [14]:
anomaly_mask = (df['age'] > 100) | (df['emp_length'] > df['age'])
anomaly_ids = df.loc[anomaly_mask, 'id'].tolist()
display(df.loc[anomaly_mask])
df = df.loc[~anomaly_mask].copy()
print(f'Удалено аномальных наблюдений: {len(anomaly_ids)}; осталось: {len(df)}.')
display(df.dtypes.rename('Исходный тип').to_frame())

,id,age,income,home,emp_length,intent,amount,rate,status,percent_income,default,cred_length
0,0,22.00,59000,RENT,123.00,PERSONAL,35000,16.02,1,0.59,Y,3
81,81,144.00,250000,RENT,4.00,VENTURE,4800,13.57,0,0.02,N,3
183,183,144.00,200000,MORTGAGE,4.00,EDUCATION,6000,11.86,0,0.03,N,2
210,210,21.00,192000,MORTGAGE,123.00,VENTURE,20000,6.54,0,0.10,N,4
575,575,123.00,80004,RENT,2.00,EDUCATION,20400,10.25,0,0.25,N,3


Удалено аномальных наблюдений: 5; осталось: 571.


,Исходный тип
id,int64
age,float64
income,int64
home,object
emp_length,float64
intent,object
amount,int64
rate,float64
status,int64
percent_income,float64


После удаления аномалий и пропусков `age` и `emp_length` содержат только целые значения, поэтому преобразуются в `int64`.
`home`, `intent` и `default` принимают значения из небольших множеств и преобразуются в `category`. Бинарный код `status` также представляется категорией; смысл кодов 0/1 не переопределяется.
`rate` и `percent_income` сохраняют тип `float64`, поскольку содержат дробные значения. Идентификатор, доход, сумма кредита и длительность кредитной истории сохраняют `int64`.

In [15]:
df[['age', 'emp_length']] = df[['age', 'emp_length']].astype('int64')
for column in ['home', 'intent', 'default', 'status']:
    df[column] = df[column].astype('category')
display(df.dtypes.rename('Итоговый тип').to_frame())
processing_steps = pd.DataFrame({
    'Этап': ['Исходный набор', 'Удаление пропусков', 'Удаление дубликатов', 'Удаление аномалий'],
    'Число строк': [len(raw_df), rows_after_dropna, rows_after_duplicates, len(df)]
})
display(processing_steps)
print(f'Удалено строк: {len(raw_df) - len(df)} ({(len(raw_df) - len(df)) / len(raw_df):.2%}).')
print(f'Потеря уникальных наблюдений: {(650 - len(df)) / 650:.2%}.')

,Итоговый тип
id,int64
age,int64
income,int64
home,category
emp_length,int64
intent,category
amount,int64
rate,float64
status,category
percent_income,float64


,Этап,Число строк
0,Исходный набор,652
1,Удаление пропусков,578
2,Удаление дубликатов,576
3,Удаление аномалий,571


Удалено строк: 81 (12.42%).
Потеря уникальных наблюдений: 12.15%.


Итоговая таблица содержит 571 строку и 12 столбцов. Удалена 81 строка, включая две лишние копии; потеря уникальных наблюдений составляет 79 из 650, или 12,15 %. Все последующие агрегаты рассчитаны по одной и той же очищенной таблице.

### 6. Группировка данных

#### Задание 1

Выполнить группировку статуса домовладения `home` по количеству значений признака `default`. Для каждой категории жилья подсчитываются записи отдельно с `Y` и `N`. Метод `size` считает строки группы; `unstack` размещает значения признака дефолта в отдельных столбцах.

In [16]:
group1 = (df.groupby(['home', 'default'], observed=True).size()
          .unstack(fill_value=0).reindex(columns=['Y', 'N']))
display(group1)

default,Y,N
home,,
MORTGAGE,26,86
OTHER,2,0
OWN,12,71
RENT,94,280


Наибольшая группа — арендаторы (`RENT`): 374 наблюдения, из них 94 с `Y` и 280 с `N`. Для `MORTGAGE` получено 26 и 86, для `OWN` — 12 и 71 соответственно.
В категории `OTHER` обе записи имеют `Y`, однако размер группы слишком мал для обобщений. Всего в подвыборке 134 записи с `Y` и 437 с `N`.
Абсолютное число записей с `Y` зависит от размера категории и само по себе не доказывает влияние формы домовладения на кредитный риск.

#### Задание 2

Выполнить группировку цели кредита `intent` по количеству статуса домовладения `home`, сформировать DataFrame со столбцом `count` и отсортировать его по убыванию.
Для подробного представления подсчитываются частоты каждой пары `intent` и `home`. Дополнительно приведено буквальное прочтение `count(home)` для каждой цели кредита: поскольку пропусков больше нет, оно равно числу заявок соответствующей цели. При равных частотах пары упорядочиваются по названиям категорий.

In [17]:
group2_pairs = (df.groupby(['intent', 'home'], observed=True).size()
                .rename('count').reset_index()
                .sort_values(['count', 'intent', 'home'], ascending=[False, True, True]))
display(group2_pairs.reset_index(drop=True))

,intent,home,count
0,EDUCATION,RENT,88
1,VENTURE,RENT,65
2,DEBTCONSOLIDATION,RENT,64
3,PERSONAL,RENT,63
4,MEDICAL,RENT,61
5,HOMEIMPROVEMENT,RENT,33
6,VENTURE,MORTGAGE,25
7,EDUCATION,MORTGAGE,24
8,PERSONAL,OWN,20
9,MEDICAL,MORTGAGE,19


In [18]:
group2_totals = (df.groupby('intent', observed=True)['home'].count()
                 .rename('count').sort_values(ascending=False).to_frame())
display(group2_totals)

,count
intent,
EDUCATION,126
VENTURE,106
PERSONAL,101
MEDICAL,92
DEBTCONSOLIDATION,87
HOMEIMPROVEMENT,59


Самая частая пара — EDUCATION и RENT (88 записей), затем VENTURE и RENT (65), DEBTCONSOLIDATION и RENT (64).
По общему числу заявок лидирует EDUCATION — 126, далее следуют VENTURE — 106, PERSONAL — 101, MEDICAL — 92, DEBTCONSOLIDATION — 87 и HOMEIMPROVEMENT — 59.
Распределение отражает структуру предоставленной очищенной подвыборки; оно не является оценкой спроса всего рынка кредитования.

#### Задание 3

Создать сводную таблицу со средним доходом `income` для каждого возраста `age`, отсортировать по убыванию среднего и округлить значения до двух знаков. `pivot_table` объединяет наблюдения одного возраста и применяет к доходу функцию `mean`.

In [19]:
pivot3 = (df.pivot_table(index='age', values='income', aggfunc='mean', observed=True)
          .sort_values('income', ascending=False).round(2))
display(pivot3)

,income
age,
26,125642.78
25,110021.18
24,89681.13
23,86162.54
22,61425.70
21,41783.50


Средний доход максимален у заёмщиков 26 лет — 125 642,78; минимален у заёмщиков 21 года — 41 783,50. В этой подвыборке средний доход возрастает вместе с возрастом от 21 до 26 лет. Наблюдаемая связь не доказывает причинного влияния возраста; среднее чувствительно к высоким доходам и составу групп.

#### Задание 4

Создать сводную таблицу со средней суммой кредита `amount`: строки — цель кредита `intent`, столбцы — возраст `age`. Отсортировать строки по возрастанию `intent` и округлить значения до двух знаков. Каждая ячейка содержит среднее только для соответствующего сочетания цели и возраста.

In [20]:
pivot4 = (df.pivot_table(index='intent', columns='age', values='amount', aggfunc='mean', observed=True)
          .sort_index().round(2))
display(pivot4)

age,21,22,23,24,25,26
intent,,,,,,
DEBTCONSOLIDATION,10583.33,18285.29,20849.14,20150.00,18603.33,23507.69
EDUCATION,13500.00,20487.50,20586.67,20355.00,19628.41,21758.33
HOMEIMPROVEMENT,4693.75,10216.67,4000.00,18904.76,21266.67,21408.33
MEDICAL,10666.67,18418.75,19323.81,21652.27,19284.00,20011.11
PERSONAL,3633.33,16908.82,18435.87,21205.00,21530.43,18400.00
VENTURE,8993.18,16704.69,22731.67,19826.04,18888.10,18850.00


Максимальное среднее среди сочетаний цели и возраста — 23 507,69 у заёмщиков 26 лет с целью DEBTCONSOLIDATION. Минимальное — 3 633,33 у заёмщиков 21 года с целью PERSONAL. Для 23-летних с целью HOMEIMPROVEMENT средняя сумма равна 4 000,00; небольшие группы могут давать нестабильные оценки. Изменение среднего по возрастам неодинаково для разных целей, поэтому единую возрастающую зависимость для всех строк утверждать нельзя.

Итоговый контроль проверяет согласованность числа строк, отсутствие пропусков и дубликатов, выполнение выбранного критерия аномалий и совпадение сумм группировок с объёмом очищенной таблицы. Очищенные данные и результаты сохраняются в папку `utility` рядом с ноутбуком.

In [21]:
assert raw_df.shape == (652, 12) and raw_df['Home'].eq('RENET').sum() == 1
assert (rows_after_dropna, rows_after_duplicates, len(df)) == (578, 576, 571)
assert not df.isna().any().any() and not df.duplicated().any()
assert df['id'].is_unique
assert (df['age'] <= 100).all() and (df['emp_length'] <= df['age']).all()
assert int(group1.to_numpy().sum()) == len(df)
assert int(group2_pairs['count'].sum()) == len(df)
assert int(group2_totals['count'].sum()) == len(df)
assert pivot3['income'].is_monotonic_decreasing
assert pivot4.index.is_monotonic_increasing

export_dir = Path('utility')
export_dir.mkdir(exist_ok=True)
df.to_csv(export_dir / 'credit_risk_clean.csv', index=False)
results = {
    'metrics': {
        'rows_raw': len(raw_df), 'columns': len(raw_df.columns),
        'rows_with_missing': rows_with_missing,
        'missing_initial': {name: int(value) for name, value in missing_initial.items()},
        'rows_after_dropna': rows_after_dropna,
        'duplicates_removed': duplicate_count,
        'rows_after_drop_duplicates': rows_after_duplicates,
        'anomaly_ids': anomaly_ids, 'rows_clean': len(df),
        'rows_removed': len(raw_df) - len(df),
        'rows_removed_percent': (len(raw_df) - len(df)) / len(raw_df) * 100,
        'unique_observations_removed': 650 - len(df),
        'unique_observations_removed_percent': (650 - len(df)) / 650 * 100,
        'percent_income_discrepancies': len(share_discrepancies),
        'missing_clean': int(df.isna().sum().sum()),
        'duplicates_clean': int(df.duplicated().sum())
    },
    'group1': group1.reset_index().to_dict(orient='records'),
    'group2_pairs': group2_pairs.to_dict(orient='records'),
    'group2_totals': group2_totals.reset_index().to_dict(orient='records'),
    'pivot3': pivot3.reset_index().to_dict(orient='records'),
    'pivot4': pivot4.reset_index().to_dict(orient='records'),
    'cleaned_dtypes': {column: str(dtype) for column, dtype in df.dtypes.items()}
}
(export_dir / 'results.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print('Контроль выполнен. Сохранены utility/credit_risk_clean.csv и utility/results.json.')

Контроль выполнен. Сохранены utility/credit_risk_clean.csv и utility/results.json.


### Вывод

В работе выполнен предварительный анализ набора кредитных заявок credit_risk.csv, содержащего 652 записи и 12 признаков. Описаны характеристики заёмщиков, параметры кредита и сведения о кредитной истории. С помощью методов head, info и describe исследованы структура таблицы, типы и распределения числовых значений. Названия столбцов приведены к нижнему регистру. Выявлены пропуски стажа и ставки, повторяющиеся записи, неоднозначные написания категорий и аномальные значения возраста и стажа.

При предобработке удалены 74 неполных наблюдения, две лишние копии записи и пять строк, не соответствующих выбранным критериям возраста и стажа. Исправлены категории RENET и No, приведены целочисленные и категориальные типы. Получена таблица из 571 записи без пропусков и дубликатов. Искусственные значения не подставлялись. Потеря 12,15 % уникальных наблюдений ограничивает интерпретацию результатов: случайность пропусков не доказана, а порог возраста 100 лет является исследовательским решением. Обнаруженные 17 расхождений доли кредита с отношением суммы к доходу отмечены без неподтверждённого исправления.

Группировки показали преобладание арендаторов: 374 записи, включая 94 с признаком дефолта Y. Наиболее частая цель кредита — образование, 126 заявок; самая частая пара цели и домовладения — EDUCATION и RENT, 88 записей. Средний доход максимален в группе 26-летних и составляет 125 642,78, минимален у 21-летних — 41 783,50. Средняя сумма кредита по сочетаниям цели и возраста изменяется от 3 633,33 для PERSONAL в возрасте 21 года до 23 507,69 для DEBTCONSOLIDATION в возрасте 26 лет. Эти результаты описывают очищенную подвыборку и не подтверждают причинных зависимостей либо закономерностей генеральной совокупности.

### Дополнительное задание

Дополнительное задание выполняется после защиты основной лабораторной работы по указанию преподавателя; в текущую работу не включено.